### Kepler's Equations and Position Estimation

#### Using the Orbit object to convert position and velocity to keplerian elements

In [ ]:
from math_tools import *
from graphing_utils import Graph
from orbit_package import Orbit

gr = Graph()
orb = Orbit()

# Values of r,v and mu to get elements for
mu = 4e5 # km^3/s^2
r0_vec = np.array([6e3,6e3,6e3]) # km
v0_vec = np.array([-5,5,0]) #km/s

# Finding kep elements
elements = orb.orbital_elements(r0_vec,v0_vec,mu)
orb.print_elements(elements)

### Creating an Ephemeris object for the orbit

In [ ]:
dt = np.linspace(-2*7200,2*7200,2000)
eph = orb.create_propagated_ephemeris(r0_vec,v0_vec,mu,dt)

##### Using built in Ephemeris functions to extract r,c,t and elements 

In [ ]:
all_r = eph.all_r()
all_v = eph.all_v()
all_t = eph.all_t()
all_ele = eph.all_ele()
print(all_t)

#### Plotting the results of the ephemeris
##### Several functions exist within the graph object to graph different elements of the ephemeris object. These can be referenced as shown to visualize and understand the orbits better

In [ ]:
from graphing_utils import Graph

gr = Graph()
gr.plot_eph_pos_vector(eph,'line')
nu = eph.all_nu()
gr.plot_dt_simple(dt,nu,['True Anomaly','Rad'])
gr.plot_eph_v_norm(eph)
gr.plot_eph_nu(eph)
gr.plot_eph_r_norm(eph)
gr.plot_eph_sma(eph)

In [ ]:
# Developing code for groundtracks
from astropy.time import Time 
import datetime
from astropy import units as u
from astropy.coordinates import EarthLocation, ITRS

def get_gst_angle(dt:datetime)->float:
    """ 
    This funciton will take in a datetime object in UTC and return the GST angle at that time
    """
    if type(dt) == list:
        gast = []
        for t in dt:
            time = Time(t,format='datetime',scale='utc')
            gast.append(time.sidereal_time('apparent',longitude=0*u.deg))
        gastd = [gas.to(u.deg) for gas in gast]
    elif type(dt) == datetime.datetime:
        time = Time(dt,format = 'datetime',scale = 'utc')
        gast = time.sidereal_time('apparent',longitude = 0*u.deg)
        gastd = gast.to(u.deg)
    else: 
        raise(TypeError)
    return gast,gastd

# dt_list = eph.all_t()

# gast, gastd = get_gst_angle(dt_list)

# import matplotlib.pyplot as plt
# plt.plot(gast)
# plt.show()
# plt.plot(gastd)
# plt.show()

xlist = eph.all_x()
ylist = eph.all_y()
zlist = eph.all_z()
tlist = eph.all_t()

xeci = [x*u.m for x in xlist]
yeci = [x*u.m for x in ylist]
zeci = [x*u.m for x in zlist]

latlist = []
lonlist = []
hlist = []
for i in range(len(xeci)):
    x = xeci[i]
    y = yeci[i]
    z = zeci[i]
    t = tlist[i]
    eci_location = EarthLocation.from_geocentric(x,y,z)
    astropy_time = Time(t,format = 'datetime')
    itrs_location = eci_location.get_itrs(astropy_time)
    lon,lat,height = itrs_location.earth_location.to_geodetic()
    latlist.append(lat)
    lonlist.append(lon)
    hlist.append(height)
    import matplotlib.pyplot as plt
    plt.plot(latlist,label='Lat')
    plt.plot(lonlist,label='Lon')
    plt.legend()
    plt.show()

    plt.plot(hlist)
    plt.show()